# SurvArena Quickstart

This notebook shows the intended AutoGluon-style user workflow for SurvArena:

1. Start with a pandas DataFrame
2. Point to the survival label columns
3. Fit a portfolio of survival models
4. Compare the models in a leaderboard
5. Generate risk and survival predictions


In [5]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
repo_root = Path.cwd().resolve().parent  # if notebook is in examples/
sys.path.insert(0, str(repo_root))
from src import SurvivalPredictor


## Create a small example survival dataset

In real use, replace this with your own CSV, Parquet file, or DataFrame.


In [6]:
rng = np.random.default_rng(7)
n = 250
age = rng.normal(62, 11, size=n).clip(18, 90)
albumin = rng.normal(3.8, 0.5, size=n).clip(1.5, 5.5)
creatinine = rng.lognormal(mean=0.0, sigma=0.35, size=n)
treatment = rng.choice(['A', 'B', 'C'], size=n, p=[0.4, 0.4, 0.2])
stage = rng.choice(['I', 'II', 'III'], size=n, p=[0.35, 0.4, 0.25])

linear_risk = (
    0.03 * (age - 60)
    - 0.7 * (albumin - 3.8)
    + 0.5 * np.log1p(creatinine)
    + 0.25 * (treatment == 'C')
    + 0.4 * (stage == 'III')
)

event_time = rng.exponential(scale=np.exp(2.7 - linear_risk))
censor_time = rng.exponential(scale=10.0, size=n)
observed_time = np.minimum(event_time, censor_time)
event = (event_time <= censor_time).astype(int)

df = pd.DataFrame({
    'age': age,
    'albumin': albumin,
    'creatinine': creatinine,
    'treatment': treatment,
    'stage': stage,
    'time': observed_time,
    'event': event,
})

train_df = df.sample(frac=0.8, random_state=7)
test_df = df.drop(train_df.index).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

train_df.head()


,age,albumin,creatinine,treatment,stage,time,event
0,72.625138,2.933434,1.433514,B,I,2.947289,1
1,45.351562,3.989714,0.890860,C,I,1.112610,0
2,61.940605,3.503497,1.226851,A,II,0.630416,0
3,59.829873,3.176512,0.918942,A,I,2.319797,1
4,52.203490,3.823560,1.624246,B,II,3.382174,0


## Fit the predictor

This is the simplest SurvArena interface.


In [7]:
predictor = SurvivalPredictor(
    label_time='time',
    label_event='event',
    presets='fast',
    eval_metric='harrell_c',
    random_state=7,
)

predictor.fit(train_df, test_data=test_df, dataset_name='notebook_demo')


/Users/justin/Documents/SurvArena/survArena/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-21 15:38:02,479] A new study created in memory with name: no-name-f54fe399-afcb-408f-832d-b81a5da7cc39
[I 2026-03-21 15:38:02,502] Trial 0 finished with value: 0.6049301226933798 and parameters: {'alpha': 2.869787477867182e-06}. Best is trial 0 with value: 0.6049301226933798.
[I 2026-03-21 15:38:02,526] Trial 1 finished with value: 0.6049301226933798 and parameters: {'alpha': 0.04780934055777077}. Best is trial 0 with value: 0.6049301226933798.
[I 2026-03-21 15:38:03,114] A new study created in memory with name: no-name-d08ef85a-9395-47f4-ab8f-fe8eba458b29
[I 2026-03-21 15:38:03,390] Trial 0 finished with value: 0.582896868387858 and parameters: {'n_estimators': 168, 'max_depth_is_none': True, 'min_sam

## Compare candidate models

This leaderboard is the SurvArena equivalent of a simple AutoML model comparison table.


In [8]:
predictor.leaderboard()


,method_id,validation_score,fit_time_sec,n_trials_completed,status,error,params
0,coxph,0.604930,1.43372,2,success,None,{'alpha': 2.869787477867182e-06}
1,rsf,0.582897,1.32354,2,success,None,"{'n_estimators': 168, 'max_depth': None, 'min_..."


## View the fit summary


In [9]:
predictor.fit_summary()


{'best_method_id': 'coxph',
 'best_params': {'alpha': 2.869787477867182e-06},
 'eval_metric': 'harrell_c',
 'preset': 'fast',
 'portfolio': ['coxph', 'rsf'],
 'test_metrics': {'uno_c': 0.6073456406593323,
  'harrell_c': 0.6815920472145081,
  'ibs': 0.1720653623342514,
  'td_auc_25': 0.8542662262916565,
  'td_auc_50': 0.673979640007019,
  'td_auc_75': 0.6529111862182617},
 'artifact_dir': 'results/predictor/notebook_demo'}

## Predict risk scores


In [10]:
risk_scores = predictor.predict_risk(test_df)
risk_scores[:10]


array([ 0.38988659,  0.02906582, -0.1732087 , -0.70644647, -0.28970371,
       -0.04500766, -0.37689075, -0.40409377, -1.02780708,  0.94159983])

## Predict survival curves


In [11]:
survival_df = predictor.predict_survival(test_df)
survival_df.head()


,t_0.315605,t_0.545672,t_0.77574,t_1.00581,t_1.23587,t_1.46594,t_1.69601,t_1.92608,t_2.15614,t_2.38621,...,t_9.51831,t_9.74837,t_9.97844,t_10.2085,t_10.4386,t_10.6686,t_10.8987,t_11.1288,t_11.3588,t_11.5889
0,0.957200,0.942374,0.919610,0.858858,0.804839,0.750382,0.712304,0.673341,0.673341,0.624738,...,0.171442,0.158838,0.158838,0.158838,0.145122,0.131869,0.119394,0.119394,0.119394,0.119394
1,0.969967,0.959469,0.943252,0.899366,0.859546,0.818576,0.789392,0.759035,0.759035,0.720411,...,0.292483,0.277321,0.277321,0.277321,0.260400,0.243583,0.227280,0.227280,0.227280,0.227280
2,0.975399,0.966767,0.953398,0.917006,0.883703,0.849142,0.824330,0.798343,0.798343,0.765000,...,0.366331,0.350743,0.350743,0.350743,0.333161,0.315479,0.298122,0.298122,0.298122,0.298122
3,0.985492,0.980366,0.972389,0.950437,0.930032,0.908516,0.892846,0.876222,0.876222,0.854563,...,0.554783,0.540809,0.540809,0.540809,0.524735,0.508212,0.491616,0.491616,0.491616,0.491616
4,0.978074,0.970367,0.958415,0.925784,0.895800,0.864550,0.842030,0.818361,0.818361,0.787870,...,0.409103,0.393572,0.393572,0.393572,0.375964,0.358151,0.340559,0.340559,0.340559,0.340559


## Using GBSG2 Dataset


In [12]:
# SurvArena Quickstart on GBSG2 dataset

import pandas as pd
import numpy as np
import sys
from pathlib import Path


In [ ]:
# Fit survival predictor on GBSG2 dataset

predictor = SurvivalPredictor(
    label_time='time',
    label_event='event',
    presets='medium',
)

predictor.fit(
    train_df,
    test_data=test_df,
    dataset_name='gbsg2_standard_v1',
)

predictor.leaderboard()


[I 2026-03-21 15:39:50,938] A new study created in memory with name: no-name-e3326d26-f25a-4d37-ab41-b0fd5d79a010
[I 2026-03-21 15:39:50,960] Trial 0 finished with value: 0.6096159219741821 and parameters: {'alpha': 0.0019628224813442808}. Best is trial 0 with value: 0.6096159219741821.
[I 2026-03-21 15:39:50,986] Trial 1 finished with value: 0.6096159219741821 and parameters: {'alpha': 0.019549524484259877}. Best is trial 0 with value: 0.6096159219741821.
[I 2026-03-21 15:39:51,012] Trial 2 finished with value: 0.6096159219741821 and parameters: {'alpha': 0.004135997393839894}. Best is trial 0 with value: 0.6096159219741821.
[I 2026-03-21 15:39:51,038] Trial 3 finished with value: 0.6096159219741821 and parameters: {'alpha': 0.0018590843630169633}. Best is trial 0 with value: 0.6096159219741821.
[I 2026-03-21 15:39:51,066] Trial 4 finished with value: 0.6096159219741821 and parameters: {'alpha': 0.0003482802087028331}. Best is trial 0 with value: 0.6096159219741821.
[I 2026-03-21 15:3

,method_id,validation_score,fit_time_sec,n_trials_completed,status,error,params
0,deepsurv,0.615327,1.511401,8,success,None,"{'hidden_layers': '256', 'activation': 'gelu',..."
1,coxph,0.609910,0.248566,8,success,None,{'alpha': 0.22420123713724419}
2,coxnet,0.609332,0.094440,8,success,None,"{'alpha': 0.0003482802087028331, 'l1_ratio': 0..."
3,rsf,0.573801,5.717277,8,success,None,"{'n_estimators': 229, 'max_depth': None, 'min_..."


## Plot Kaplan-Meier comparison

This compares empirical Kaplan-Meier curves against the model's mean predicted survival within risk groups.


In [ ]:
ax = predictor.plot_kaplan_meier_comparison(test_df)
ax
